# GEC Project — Notebook 2+3: LSTM Without Attention & LSTM With Attention

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Embedding,
    LSTM,
    Bidirectional,
    Dropout
)

print("TensorFlow Version:", tf.__version__)

2026-05-18 08:46:48.839606: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779094009.013821      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779094009.066992      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779094009.443110      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779094009.443153      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779094009.443156      57 computation_placer.cc:177] computation placer alr

TensorFlow Version: 2.19.0


## Load Data

In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:5]:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/dariocioni/c4200m/C4_200M.tsv-00003-of-00010
/kaggle/input/datasets/dariocioni/c4200m/C4_200M.tsv-00004-of-00010
/kaggle/input/datasets/dariocioni/c4200m/C4_200M.tsv-00002-of-00010
/kaggle/input/datasets/dariocioni/c4200m/C4_200M.tsv-00001-of-00010
/kaggle/input/datasets/dariocioni/c4200m/C4_200M.tsv-00009-of-00010


In [3]:
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/saved_data', exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [4]:
file_path = "/kaggle/input/datasets/dariocioni/c4200m/C4_200M.tsv-00001-of-00010"

df = pd.read_csv(
    file_path,
    sep='\t',
    header=None,
    names=['incorrect', 'correct']
)

df.head()

,incorrect,correct
0,I think I'm goign to have to inform your posts...,I think I'm going to have to re-read your post...
1,EverySaturday we have the King Of Club 'Drawin...,Every Saturday we have the King Of Clubs Drawing.
2,Just then background of lemonade stand was ink...,Then background of the lemonade stand was inke...
3,V-029595 Aleter Universe Lip Gloss Lapstick Qu...,V-029595 Altered Universe Lip Gloss Lipstick Q...
4,In the same floor is a staircase that access a...,On the same floor there is a staircase that gi...


# **Working on 1 million samples**

In [5]:
df = df.dropna()

print("\n New Shape:", df.shape)

df_small = df.sample(
    n=1_000_000,
    random_state=42
)

df_BOW = df.sample(
    n=500_000,
    random_state=42
)


print("Sampled Dataset Shape:", df_small.shape)
print("BOW Dataset Shape:", df_BOW.shape)


save_path = "/kaggle/working/saved_data/gec_1m.csv"
BOW_save_path = "/kaggle/working/saved_data/gec_500k.csv"


df_small.to_csv(save_path, index=False)
df_BOW.to_csv(BOW_save_path, index=False)


print("Dataset saved successfully.")
print("Saved at:", save_path)

print("BOW Dataset saved successfully.")
print("Saved at:", BOW_save_path)


 New Shape: (18376963, 2)
Sampled Dataset Shape: (1000000, 2)
BOW Dataset Shape: (500000, 2)
Dataset saved successfully.
Saved at: /kaggle/working/saved_data/gec_1m.csv
BOW Dataset saved successfully.
Saved at: /kaggle/working/saved_data/gec_500k.csv


## Data Cleaning

In [6]:
import re

def clean_text(text):
    
    # convert to string
    text = str(text)
    
    # lowercase
    text = text.lower()
    
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    
    # remove leading/trailing spaces
    text = text.strip()
    
    return text


# Apply cleaning
df_small['incorrect'] = df_small['incorrect'].apply(clean_text)
df_small['correct'] = df_small['correct'].apply(clean_text)


df_BOW['incorrect'] = df_BOW['incorrect'].apply(clean_text)
df_BOW['correct'] = df_BOW['correct'].apply(clean_text)



print("Text cleaning completed.")


#Check
for i in range(5):
    print("Incorrect :", df_small.iloc[i]['incorrect'])
    print("Correct   :", df_small.iloc[i]['correct'])
    print("-" * 60)


#Save
m_clean_path = "/kaggle/working/saved_data/gec_1m_cleaned.csv"
BOW_clean_path = "/kaggle/working/saved_data/gec_500k_cleaned.csv"



df_small.to_csv(m_clean_path, index=False)
df_BOW.to_csv(BOW_clean_path, index=False)



print("Cleaned dataset saved successfully.")

Text cleaning completed.
Incorrect : to snort a sticky phlegm from his nostrils.
Correct   : to snort the sticky phlegm out of his nostrils.
------------------------------------------------------------
Incorrect : this downward trend is not looking good.i was asked by a tpu wcg member if i was going to stay or return to xs and i said i would stay unless tpu commitment started to wane.
Correct   : this downward trend is not looking good.i was asked by a tpu wcg member if i was going to stay or return to xs and i said i would stay,unless tpu commitment started to wane.
------------------------------------------------------------
Incorrect : to keep your roof clean and health, visit to jim thomas maintenance!
Correct   : to keep your roof clean and healthy, visit jim thomas maintenance!
------------------------------------------------------------
Incorrect : ecommerce best practices optimizing the user experience ecommerce ux overview and best practices well-performanceing e-c sites have 

## **Tokenization**  

In [7]:
input_texts = df_small['incorrect'].tolist()
target_texts = df_small['correct'].tolist()

print("Number of Input Sentences :", len(input_texts))
print("Number of Target Sentences:", len(target_texts))

Number of Input Sentences : 1000000
Number of Target Sentences: 1000000


In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer

VOCAB_SIZE = 30000

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(
    input_texts + target_texts
)

print("Tokenizer created successfully.")
print("Vocabulary Size:", VOCAB_SIZE)
print("Vocabulary Size:", VOCAB_SIZE)

Tokenizer created successfully.
Vocabulary Size: 30000
Vocabulary Size: 30000


**Save Tokenizer**

In [9]:
import pickle

tokenizer_path = "/kaggle/working/checkpoints/tokenizer.pkl"

with open(tokenizer_path, 'wb') as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved successfully.")

Tokenizer saved successfully.


# LSTM Without Attention

In [10]:
input_sequences = tokenizer.texts_to_sequences(input_texts)
target_sequences = tokenizer.texts_to_sequences(target_texts)

print("Example Input Sequence:")
print(input_sequences[0])

print("\nExample Target Sequence:")
print(target_sequences[0])

Example Input Sequence:
[4, 1, 6, 8188, 1, 22, 50, 1]

Example Target Sequence:
[4, 1, 2, 8188, 1, 53, 5, 50, 1]


In [11]:
max_input_len = max(
    len(seq) for seq in input_sequences
)

max_target_len = max(
    len(seq) for seq in target_sequences
)

print("Max Input Length :", max_input_len)
print("Max Target Length:", max_target_len)

Max Input Length : 5378
Max Target Length: 187


In [12]:
MAX_LEN = 50

filtered_input_sequences = []
filtered_target_sequences = []

for inp, tgt in zip(input_sequences, target_sequences):
    
    if len(inp) <= MAX_LEN and len(tgt) <= MAX_LEN:
        
        filtered_input_sequences.append(inp)
        filtered_target_sequences.append(tgt)

print("Remaining Samples:", len(filtered_input_sequences))

input_sequences = filtered_input_sequences
target_sequences = filtered_target_sequences

max_input_len = max(len(seq) for seq in input_sequences)
max_target_len = max(len(seq) for seq in target_sequences)

print("Max Input Length :", max_input_len)
print("Max Target Length:", max_target_len)

Remaining Samples: 939108
Max Input Length : 50
Max Target Length: 50


In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

encoder_input_data = pad_sequences(
    input_sequences,
    maxlen=max_input_len,
    padding='post'
)

decoder_input_data = pad_sequences(
    target_sequences,
    maxlen=max_target_len,
    padding='post'
)

print("Encoder Input Shape:", encoder_input_data.shape)
print("Decoder Input Shape:", decoder_input_data.shape)

Encoder Input Shape: (939108, 50)
Decoder Input Shape: (939108, 50)


In [14]:
decoder_target_data = np.zeros_like(decoder_input_data)

decoder_target_data[:, :-1] = decoder_input_data[:, 1:]

print("Decoder Target Shape:")
print(decoder_target_data.shape)

Decoder Target Shape:
(939108, 50)


In [15]:
from sklearn.model_selection import train_test_split

(
    encoder_train,
    encoder_test,
    
    decoder_input_train,
    decoder_input_test,
    
    decoder_target_train,
    decoder_target_test

) = train_test_split(
    
    encoder_input_data,
    decoder_input_data,
    decoder_target_data,
    
    test_size=0.2,
    random_state=42
)

print("Encoder Train Shape:", encoder_train.shape)
print("Encoder Test Shape :", encoder_test.shape)

Encoder Train Shape: (751286, 50)
Encoder Test Shape : (187822, 50)


In [16]:
encoder_train = np.array(encoder_train)
encoder_test = np.array(encoder_test)

decoder_input_train = np.array(decoder_input_train)
decoder_input_test = np.array(decoder_input_test)

decoder_target_train = np.array(decoder_target_train)
decoder_target_test = np.array(decoder_target_test)

print("Conversion completed.")

Conversion completed.


In [17]:
decoder_target_train = np.expand_dims(
    decoder_target_train,
    -1
)

decoder_target_test = np.expand_dims(
    decoder_target_test,
    -1
)

print("Decoder Target Shape:")
print(decoder_target_train.shape)

Decoder Target Shape:
(751286, 50, 1)


In [18]:
VOCAB_SIZE = 30000

EMBEDDING_DIM = 256

LSTM_UNITS = 256

BATCH_SIZE = 128

EPOCHS = 5

print("Done")

Done


In [19]:
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Embedding,
    Dense
)

from tensorflow.keras.models import Model

# Encoder input
encoder_inputs = Input(shape=(max_input_len,))

# Embedding layer
encoder_embedding = Embedding(
    VOCAB_SIZE,
    EMBEDDING_DIM
)(encoder_inputs)

# Encoder LSTM
encoder_lstm = LSTM(
    LSTM_UNITS,
    return_state=True
)

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_embedding
)

encoder_states = [state_h, state_c]

print("Done")

I0000 00:00:1779094286.979838      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Done


In [20]:
decoder_inputs = Input(shape=(max_target_len,))

decoder_embedding = Embedding(
    VOCAB_SIZE,
    EMBEDDING_DIM
)(decoder_inputs)

decoder_lstm = LSTM(
    LSTM_UNITS,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(
    VOCAB_SIZE,
    activation='softmax'
)

decoder_outputs = decoder_dense(
    decoder_outputs
)
print("Done")

Done


In [21]:
seq2seq_model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

seq2seq_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

seq2seq_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 256)   │  7,680,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 50, 256)   │  7,680,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    525,312 │ embedding[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 50, 256), │    525,312 │ embedding_1[0][0… │
│                     │ (None, 256),      │            │ lstm[0][1],       │
│                     │ (None, 256)]      │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 30000) │  7,710,000 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,120,624 (92.01 MB)

 Trainable params: 24,120,624 (92.01 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
checkpoint_path = "/kaggle/working/checkpoints/lstm_seq2seq.keras"

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

print("Done")

Done


In [ ]:
history = seq2seq_model.fit(
    
    [encoder_train, decoder_input_train],
    decoder_target_train,
    
    validation_split=0.1,
    
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    
    callbacks=[
        checkpoint_callback,
        early_stop
    ]
)

Epoch 1/5


I0000 00:00:1779094292.316379     131 cuda_dnn.cc:529] Loaded cuDNN version 91002


5283/5283 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step - accuracy: 0.6643 - loss: 2.6907
Epoch 1: val_accuracy improved from -inf to 0.71095, saving model to /kaggle/working/checkpoints/lstm_seq2seq.keras
5283/5283 ━━━━━━━━━━━━━━━━━━━━ 1515s 286ms/step - accuracy: 0.6643 - loss: 2.6906 - val_accuracy: 0.7110 - val_loss: 1.9891
Epoch 2/5
2948/5283 ━━━━━━━━━━━━━━━━━━━━ 10:15 264ms/step - accuracy: 0.7163 - loss: 1.9112

In [ ]:
# =========================
# Encoder Inference Model
# =========================

encoder_model = Model(
    encoder_inputs,
    encoder_states
)
# =========================
# Decoder Inference Model
# =========================

# Decoder state inputs
decoder_state_input_h = Input(shape=(LSTM_UNITS,))
decoder_state_input_c = Input(shape=(LSTM_UNITS,))

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

# Decoder embedding
decoder_embedding2 = Embedding(
    VOCAB_SIZE,
    EMBEDDING_DIM
)(decoder_inputs)

# Decoder LSTM
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_embedding2,
    initial_state=decoder_states_inputs
)

decoder_states2 = [state_h2, state_c2]

# Dense output
decoder_outputs2 = decoder_dense(
    decoder_outputs2
)

# Final decoder model
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    
    [decoder_outputs2] + decoder_states2
)


# =========================
# Reverse Token Mapping
# =========================

reverse_word_index = {
    index: word
    for word, index in tokenizer.word_index.items()
}

print("Reverse dictionary created.")


# =========================
# Decode Sequence Function
# =========================

def decode_sequence(input_seq):
    
    # Encode input sentence
    states_value = encoder_model.predict(
        input_seq,
        verbose=0
    )
    
    # Start token
    target_seq = np.zeros((1, 1))
    
    target_seq[0, 0] = tokenizer.word_index['start']
    
    stop_condition = False
    
    decoded_sentence = ''
    
    while not stop_condition:
        
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )
        
        # Highest probability token
        sampled_token_index = np.argmax(
            output_tokens[0, -1, :]
        )
        
        sampled_word = reverse_word_index.get(
            sampled_token_index,
            ''
        )
        
        # Stop conditions
        if (
            sampled_word == 'end'
            or len(decoded_sentence.split()) > max_target_len
        ):
            stop_condition = True
        
        else:
            decoded_sentence += ' ' + sampled_word
        
        # Update target sequence
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        
        # Update states
        states_value = [h, c]
    
    return decoded_sentence

In [ ]:
for i in range(5):
    
    input_seq = encoder_test[i:i+1]
    
    predicted_sentence = decode_sequence(
        input_seq
    )
    
    original_input = input_texts[i]
    
    original_target = target_texts[i]
    
    print("Incorrect Sentence:")
    print(original_input)
    
    print("\nActual Correction:")
    print(original_target)
    
    print("\nPredicted Correction:")
    print(predicted_sentence)
    
    print("\n" + "="*80 + "\n")

## **LSTM With Attention**

In [ ]:
from tensorflow.keras.layers import Attention

# =========================
# Hyperparameters
# =========================

VOCAB_SIZE = 30000

EMBEDDING_DIM = 256

LSTM_UNITS = 256

BATCH_SIZE = 125

EPOCHS = 5

In [ ]:
encoder_inputs = Input(
    shape=(max_input_len,)
)

encoder_embedding = Embedding(
    VOCAB_SIZE,
    EMBEDDING_DIM
)(encoder_inputs)

encoder_lstm = LSTM(
    LSTM_UNITS,
    return_sequences=True,
    return_state=True
)

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_embedding
)

encoder_states = [state_h, state_c]

In [ ]:
decoder_inputs = Input(
    shape=(max_target_len,)
)

decoder_embedding = Embedding(
    VOCAB_SIZE,
    EMBEDDING_DIM
)(decoder_inputs)

decoder_lstm = LSTM(
    LSTM_UNITS,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

print("Done")

**attention_layer**

In [ ]:
attention_layer = Attention()

attention_output = attention_layer(
    [decoder_outputs, encoder_outputs]
)

# =========================
# Concatenate Attention = Attention + Decoder
# =========================

from tensorflow.keras.layers import Concatenate

decoder_combined_context = Concatenate(
    axis=-1
)(
    [decoder_outputs, attention_output]
)

print("Done")

In [ ]:
decoder_dense = Dense(
    VOCAB_SIZE,
    activation='softmax'
)

decoder_final_output = decoder_dense(
    decoder_combined_context
)
print("Done")

In [ ]:
attention_model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_final_output
)

attention_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

attention_model.summary()
print("Done")

In [ ]:
checkpoint_path = "/kaggle/working/checkpoints/attention_model.keras"

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)
print("Done")

In [ ]:
history_attention = attention_model.fit(
    
    [encoder_train, decoder_input_train],
    decoder_target_train,
    
    validation_split=0.1,
    
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    
    callbacks=[
        checkpoint_callback,
        early_stop
    ]
)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy
plt.figure(figsize=(10, 5))

plt.plot(
    history_attention.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history_attention.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title('Attention Model Accuracy')

plt.xlabel('Epoch')

plt.ylabel('Accuracy')

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history_attention.history['loss'],
    label='Training Loss'
)

plt.plot(
    history_attention.history['val_loss'],
    label='Validation Loss'
)

plt.title('Attention Model Loss')

plt.xlabel('Epoch')

plt.ylabel('Loss')

plt.legend()

plt.show()

In [ ]:
# =========================
# Encoder Inference Model
# =========================

encoder_model_attention = Model(
    encoder_inputs,
    [encoder_outputs, state_h, state_c]
)

# =========================
# Decoder Inputs
# =========================

decoder_state_input_h = Input(
    shape=(LSTM_UNITS,)
)

decoder_state_input_c = Input(
    shape=(LSTM_UNITS,)
)

decoder_hidden_state_input = Input(
    shape=(max_input_len, LSTM_UNITS)
)

# =========================
# Decoder Embedding
# =========================

decoder_inputs_single = Input(shape=(1,))

decoder_embedding2 = Embedding(
    VOCAB_SIZE,
    EMBEDDING_DIM
)(decoder_inputs_single)

# =========================
# Decoder LSTM
# =========================

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_embedding2,
    
    initial_state=[
        decoder_state_input_h,
        decoder_state_input_c
    ]
)

# =========================
# Attention Layer
# =========================

attention_output2 = attention_layer(
    [decoder_outputs2, decoder_hidden_state_input]
)

# =========================
# Combine Attention Context
# =========================

decoder_combined_context2 = Concatenate(
    axis=-1
)(
    [decoder_outputs2, attention_output2]
)

# =========================
# Final Decoder Output
# =========================

decoder_outputs2 = decoder_dense(
    decoder_combined_context2
)

In [ ]:
decoder_model_attention = Model(
    
    [
        decoder_inputs_single,
        decoder_hidden_state_input,
        decoder_state_input_h,
        decoder_state_input_c
    ],
    
    [
        decoder_outputs2,
        state_h2,
        state_c2
    ]
)

In [ ]:
reverse_word_index = {
    index: word
    for word, index in tokenizer.word_index.items()
}

In [ ]:
def decode_sequence_attention(input_seq):
    
    # Encode input
    encoder_outs, state_h, state_c = (
        encoder_model_attention.predict(
            input_seq,
            verbose=0
        )
    )
    
    # Start token
    target_seq = np.zeros((1, 1))
    
    target_seq[0, 0] = tokenizer.word_index['start']
    
    decoded_sentence = ''
    
    stop_condition = False
    
    while not stop_condition:
        
        output_tokens, h, c = (
            decoder_model_attention.predict(
                [
                    target_seq,
                    encoder_outs,
                    state_h,
                    state_c
                ],
                verbose=0
            )
        )
        
        sampled_token_index = np.argmax(
            output_tokens[0, -1, :]
        )
        
        sampled_word = reverse_word_index.get(
            sampled_token_index,
            ''
        )
        
        # Stop condition
        if (
            sampled_word == 'end'
            or len(decoded_sentence.split()) > max_target_len
        ):
            stop_condition = True
        
        else:
            decoded_sentence += ' ' + sampled_word
        
        # Update target token
        target_seq = np.zeros((1, 1))
        
        target_seq[0, 0] = sampled_token_index
        
        # Update states
        state_h, state_c = h, c
    
    return decoded_sentence

In [ ]:
for i in range(5):
    
    input_seq = encoder_test[i:i+1]
    
    predicted_sentence = decode_sequence_attention(
        input_seq
    )
    
    print("Incorrect Sentence:")
    print(input_texts[i])
    
    print("\nActual Correction:")
    print(target_texts[i])
    
    print("\nPredicted Correction:")
    print(predicted_sentence)
    
    print("\n" + "="*80 + "\n")

#  **Side-by-Side Comparison**

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────────────
comparison = pd.DataFrame([lstm_metrics, attn_metrics])
print(comparison.to_string(index=False))

# ── Side-by-side bar chart ────────────────────────────────────────────────────
metrics_to_plot = ['accuracy', 'f1', 'precision', 'recall']
x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

bars1 = ax.bar(x - width/2,
               [lstm_metrics[m] for m in metrics_to_plot],
               width, label='LSTM (No Attention)', color='steelblue')
bars2 = ax.bar(x + width/2,
               [attn_metrics[m] for m in metrics_to_plot],
               width, label='LSTM + Attention', color='coral')

ax.set_xticks(x)
ax.set_xticklabels([m.capitalize() for m in metrics_to_plot])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('LSTM vs LSTM + Attention — All Metrics')
ax.legend()
ax.bar_label(bars1, fmt='%.4f', padding=3, fontsize=8)
ax.bar_label(bars2, fmt='%.4f', padding=3, fontsize=8)

plt.tight_layout()
plt.savefig('/kaggle/working/saved_data/lstm_comparison.png', dpi=120)
plt.show()